In [ ]:
#@title CCTV Unpaid Customer & Long-Stay Detector
# 1. Install Dependencies
!pip install ultralytics opencv-python-headless PySimpleGUI

# 2. Import Libraries
import cv2
import PySimpleGUI as sg
import threading
import time
from ultralytics import YOLO
from collections import defaultdict
import numpy as np

# 3. Configuration & Global Variables
model = YOLO('yolov8n.pt')  # Load a pre-trained YOLOv8 nano model
video_source = 0  # Use 0 for webcam, or replace with a video file path

# --- Threading and GUI Communication ---
frame_lock = threading.Lock() # To safely access the latest frame
latest_frame = None
stop_event = threading.Event()
unpaid_count = 0
long_stay_count = 0

# --- Tracking Data ---
tracked_timers = defaultdict(float) # Stores start time for each tracked ID
customer_status = defaultdict(lambda: {'paid': False, 'ordered': False})

# 4. Video Processing Thread
def video_processing_thread(long_stay_threshold, exit_line_y):
    global latest_frame, unpaid_count, long_stay_count
    cap = cv2.VideoCapture(video_source)
    if not cap.isOpened():
        sg.popup_error('Error opening video source!')
        stop_event.set()
        return

    while not stop_event.is_set():
        ret, frame = cap.read()
        if not ret:
            break
        
        # --- Detection and Tracking ---
        results = model.track(frame, persist=True, classes=[0]) # Track only 'person' class
        if results[0].boxes.id is None:
            processed_frame = frame.copy()
        else:
            processed_frame = results[0].plot() # Draw boxes and IDs
            boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)
            
            current_long_stay = 0
            for box, track_id in zip(boxes, track_ids):
                # --- Long-Stay Detection ---
                if track_id not in tracked_timers:
                    tracked_timers[track_id] = time.time()
                
                stay_duration = time.time() - tracked_timers[track_id]
                if stay_duration > long_stay_threshold and not customer_status[track_id]['ordered']:
                    current_long_stay += 1
                    cv2.putText(processed_frame, 'LONG STAY', (box[0], box[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

                # --- Unpaid Customer Detection ---
                cy = (box[1] + box[3]) // 2 # Bbox center y
                if cy > exit_line_y and not customer_status[track_id]['paid']:
                    unpaid_count += 1
                    customer_status[track_id]['paid'] = True # Mark as 'paid' to avoid double counting
                    cv2.rectangle(processed_frame, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 3) # Highlight unpaid

            long_stay_count = current_long_stay

        # Draw the exit line
        cv2.line(processed_frame, (0, exit_line_y), (processed_frame.shape[1], exit_line_y), (255, 0, 0), 2)
        cv2.putText(processed_frame, 'EXIT', (10, exit_line_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        # Update the shared frame for the GUI
        with frame_lock:
            latest_frame = cv2.imencode('.png', processed_frame)[1].tobytes()
    
    cap.release()

# 5. GUI Thread
def main_gui():
    global unpaid_count, long_stay_count

    sg.theme('DarkGrey2')

    # --- GUI Layout ---
    col1 = [
        [sg.Text('CCTV Feed')],
        [sg.Image(key='-IMAGE-', size=(640, 480))]
    ]

    col2 = [
        [sg.Text('CONTROLS & STATS', font='Any 15')],
        [sg.HSeparator()],
        [sg.Text('Long Stay Threshold (s):'), sg.Input('10', key='-LONG_STAY_THRESH-', size=(5,1))],
        [sg.Text('Exit Line Y-Position:'), sg.Input('400', key='-EXIT_LINE_Y-', size=(5,1))],
        [sg.Button('Update Settings')],
        [sg.HSeparator()],
        [sg.Text('Unpaid Customers:', size=(20,1)), sg.Text('0', key='-UNPAID_COUNT-', font='Any 15')],
        [sg.Text('Long-Stay Customers:', size=(20,1)), sg.Text('0', key='-LONG_STAY_COUNT-', font='Any 15')],
        [sg.HSeparator()],
        [sg.Text('', key='-ALERT-', text_color='red', font='Any 20', size=(25,2))]
    ]
    
    layout = [[sg.Column(col1), sg.VSeperator(), sg.Column(col2)]]
    window = sg.Window('CCTV Security Monitor', layout, finalize=True)

    # --- Start Video Thread ---
    long_stay_threshold = int(window['-LONG_STAY_THRESH-'].get())
    exit_line_y = int(window['-EXIT_LINE_Y-'].get())
    video_thread = threading.Thread(target=video_processing_thread, args=(long_stay_threshold, exit_line_y), daemon=True)
    video_thread.start()

    # --- GUI Event Loop ---
    while True:
        event, values = window.read(timeout=100) # Timeout allows the loop to run and update image

        if event == sg.WIN_CLOSED:
            break
        
        if event == 'Update Settings':
            sg.popup('Settings will be applied on next restart.', 'Please close and run the cell again.')

        # Update GUI elements from shared variables
        with frame_lock:
            if latest_frame:
                window['-IMAGE-'].update(data=latest_frame)
        
        window['-UNPAID_COUNT-'].update(unpaid_count)
        window['-LONG_STAY_COUNT-'].update(long_stay_count)

        # Check for alerts
        alert_msg = ''
        if unpaid_count > 0:
            alert_msg += 'UNPAID CUSTOMER ALERT!\n'
        if long_stay_count > 0:
            alert_msg += 'LONG STAY ALERT!'
        window['-ALERT-'].update(alert_msg)

    # --- Cleanup ---
    stop_event.set()
    video_thread.join()
    window.close()

# 6. Run the Application
if __name__ == '__main__':
    # In Colab, functions are defined but not run until called.
    # We call main_gui() to start the app.
    main_gui()